In [53]:
import time

import pandas as pd
from sklearn.metrics import roc_auc_score

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")
test = pd.read_csv("../test.csv")

feature_columns = [c for c in train.columns if c not in ["row_id", "target"]]

print("Размер train выборки:", train.shape)
print("Размер validation выборки:", validation.shape)
print("Размер test выборки:", test.shape)
print("Количество features:", len(feature_columns))



Размер train выборки: (2520, 514)
Размер validation выборки: (560, 514)
Размер test выборки: (1116, 513)
Количество features: 512


In [54]:
print("Баланс классов:")
print(train["target"].value_counts())

print("\nДоли классов:")
print(train["target"].value_counts(normalize=True))

print("\nКоличество пропусков:", train.isna().sum().sum())
print("Количество дубликатов:", train.duplicated().sum())

constant_features = [
    c for c in feature_columns
    if train[c].nunique() <= 1
]

print("Количество const признаков:", len(constant_features))

Баланс классов:
target
1    1260
0    1260
Name: count, dtype: int64

Доли классов:
target
1    0.5
0    0.5
Name: proportion, dtype: float64

Количество пропусков: 0
Количество дубликатов: 0
Количество const признаков: 0


In [55]:
def evaluate_model(name, model):
    start = time.perf_counter()

    model.fit(train[feature_columns], train["target"])

    probability = model.predict_proba(validation[feature_columns])[:, 1]

    auc = roc_auc_score(validation["target"], probability)

    elapsed = time.perf_counter() - start

    return {
        "model": name,
        "roc_auc": auc,
        "time_sec": elapsed,
    }

In [56]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 20260916

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        random_state=SEED,
    ),
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=SEED,
    ),
}

In [57]:
results = []

for name, model in models.items():
    result = evaluate_model(name, model)
    results.append(result)

results_df = (
    pd.DataFrame(results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

results_df

,model,roc_auc,time_sec
0,Gradient Boosting,0.905306,21.779472
1,Random Forest,0.903246,0.555334
2,Logistic Regression,0.883916,0.077921
3,Decision Tree,0.747309,0.267220


## Базовое сравнение

Сравнил 4 модели на одних данных

Все модели обучались на `train` 

Качество проверялось на `validation`

Сравниваю ROC-AUC и время работы

## Вывод по базовым моделям

У Decision Tree: 0.747

У Logistic Regression: 0.884

Лучшие рез-ты у ансамблей:
Random Forest: 0.905, Gradient Boosting: 0.905

Gradient Boosting чуть лучше, зато Random Forest работал быстрее

=> дальше я решил подробнее вникнуть в Random Forest

In [58]:
from sklearn.ensemble import RandomForestClassifier

rf_results = []

for max_depth in [3, 5, 10, 20, None]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF max_depth={max_depth}",
        model,
    )

    result["max_depth"] = max_depth
    rf_results.append(result)

rf_depth_df = pd.DataFrame(rf_results)
rf_depth_df

,model,roc_auc,time_sec,max_depth
0,RF max_depth=3,0.876186,0.433238,3.0
1,RF max_depth=5,0.881237,0.461969,5.0
2,RF max_depth=10,0.900038,0.598069,10.0
3,RF max_depth=20,0.903374,0.635533,20.0
4,RF max_depth=None,0.903246,0.646478,NaN


### Вывод по max_depth

При увеличении глубины ROC-AUC рос

После примерно 20 улучшение стало маленьким

Лучший рез-т - без ограничения глубины, => дальше `max_depth=None`.

In [59]:
rf_feature_results = []

for max_features in ["sqrt", "log2", 0.25, 0.5, 1.0]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features=max_features,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF max_features={max_features}",
        model,
    )

    result["max_features"] = max_features
    rf_feature_results.append(result)

rf_features_df = pd.DataFrame(rf_feature_results)
rf_features_df

,model,roc_auc,time_sec,max_features
0,RF max_features=sqrt,0.903246,0.669609,sqrt
1,RF max_features=log2,0.892105,0.391889,log2
2,RF max_features=0.25,0.901250,3.379788,0.25
3,RF max_features=0.5,0.906441,6.779062,0.5
4,RF max_features=1.0,0.907073,13.562260,1.0


### Вывод по max_features

При увеличении `max_features` качество в целом улучшалось

Лучший ROC-AUC - при `max_features=1.0`

`0.5` дало похожий рез-т, но быстрее

Далее `max_features=1.0`

In [60]:
rf_leaf_results = []

for min_samples_leaf in [1, 2, 5, 10, 20]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=min_samples_leaf,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF min_samples_leaf={min_samples_leaf}",
        model,
    )

    result["min_samples_leaf"] = min_samples_leaf
    rf_leaf_results.append(result)

rf_leaf_df = pd.DataFrame(rf_leaf_results)
rf_leaf_df

,model,roc_auc,time_sec,min_samples_leaf
0,RF min_samples_leaf=1,0.907073,14.020882,1
1,RF min_samples_leaf=2,0.908476,13.039442,2
2,RF min_samples_leaf=5,0.905918,11.459946,5
3,RF min_samples_leaf=10,0.903304,9.348527,10
4,RF min_samples_leaf=20,0.900242,7.796699,20


### Вывод по min_samples_leaf

Лучший - при `min_samples_leaf=2`

При увеличении ROC-AUC начал падать

Дальше `min_samples_leaf=2`

In [61]:
rf_estimators_results = []

for n_estimators in [50, 100, 200, 500, 1000]:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF n_estimators={n_estimators}",
        model,
    )

    result["n_estimators"] = n_estimators
    rf_estimators_results.append(result)

rf_estimators_df = pd.DataFrame(rf_estimators_results)
rf_estimators_df

,model,roc_auc,time_sec,n_estimators
0,RF n_estimators=50,0.898999,3.450339,50
1,RF n_estimators=100,0.904196,6.417516,100
2,RF n_estimators=200,0.908476,12.318257,200
3,RF n_estimators=500,0.913246,29.881926,500
4,RF n_estimators=1000,0.911626,55.281790,1000


### Вывод по n_estimators

При увеличении деревьев качество сначала росло

Лучший - при `n_estimators=500`: 0.9134

При 1000 деревьях качество упало при времени 2 раза больше

Дальше `n_estimators=500`

In [62]:
rf_criterion_results = []

for criterion in ["gini", "entropy", "log_loss"]:
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=2,
        criterion=criterion,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF criterion={criterion}",
        model,
    )

    result["criterion"] = criterion
    rf_criterion_results.append(result)

rf_criterion_df = pd.DataFrame(rf_criterion_results)
rf_criterion_df

,model,roc_auc,time_sec,criterion
0,RF criterion=gini,0.913246,29.397221,gini
1,RF criterion=entropy,0.910038,36.818005,entropy
2,RF criterion=log_loss,0.910026,36.884797,log_loss


### Вывод по criterion

Лучший - с `criterion="gini"`: 0.9134

`entropy` и `log_loss` -  дали рез-т похуже

Дальше `criterion="gini"`

In [63]:
final_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features=1.0,
    min_samples_leaf=2,
    criterion="gini",
    random_state=SEED,
    n_jobs=-1,
)

final_rf_result = evaluate_model(
    "Final Random Forest + mean features",
    final_rf,
)

pd.DataFrame([final_rf_result])

,model,roc_auc,time_sec
0,Final Random Forest + mean features,0.913246,28.931769


## Итог по Random Forest

Базовый Random Forest был с ROC-AUC - 0.9045

После подбора гиперпараметров лучший результат вырос до 0.9134

Лучшие параметры:

- `n_estimators=500`
- `max_depth=None`
- `max_features=1.0`
- `min_samples_leaf=2`
- `criterion="gini"`

На этом подбор параметров Random Forest останавливаю

In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Математические признаки для пары объектов 
мною были замечены следующие закономерности:
так как всего признаков 512 их можно представить как две половины по 256, то есть два скрытых объекта 
от 0 до 255 один и второй от 256 до 512.Можно по модулю посчитать разницу между первым и вторым и это будет показывать насколько сильно отличается признак у первого и второго объекта.


In [65]:
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier



def make_pair_features(df):
    X = df[feature_columns].to_numpy()

    # Первые 256 признаков — объект A
    A = X[:, :256]

    # Вторые 256 признаков — объект B
    B = X[:, 256:]

    # Для каждой координаты считаем расстояние:
    # |A_i - B_i|
    abs_diff = np.abs(A - B)

    # Оставляем исходные 512 признаков
    # + добавляем 256 расстояний
    X_extended = np.hstack([
        X,
        abs_diff
    ])

    return X_extended


X_train_extra = make_pair_features(train)
X_validation_extra = make_pair_features(validation)
X_test_extra = make_pair_features(test)


print("Исходных признаков:", len(feature_columns))
print("После добавления |A-B|:", X_train_extra.shape[1])

Исходных признаков: 512
После добавления |A-B|: 768


In [66]:
best_extra = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=None,
    max_features=1.0,
    min_samples_leaf=2,
    criterion="gini",
    random_state=SEED,
    n_jobs=-1,
)

best_extra.fit(
    X_train_extra,
    train["target"],
)

extra_probability = best_extra.predict_proba(
    X_validation_extra
)[:, 1]

extra_auc = roc_auc_score(
    validation["target"],
    extra_probability,
)

print("Best ExtraTrees ROC-AUC:", extra_auc)

Best ExtraTrees ROC-AUC: 0.9258673469387755


## Итог по Extra Trees

Для Extra Trees дополнительно использовала математические признаки `|A-B|`.

**ROC-AUC = 0.9259**

Лучшие параметры:

- `n_estimators=100`
- `max_depth=None`
- `max_features=1.0`
- `min_samples_leaf=2`
- `criterion="gini"`

Лучший Random Forest давал около `0.9133`.

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

best_logistic = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "model",
        LogisticRegression(
            C=0.003,
            penalty="l2",
            solver="liblinear",
            max_iter=5000,
            random_state=SEED,
        ),
    ),
])

best_logistic.fit(
    train[feature_columns],
    train["target"],
)

logistic_probability = best_logistic.predict_proba(
    validation[feature_columns]
)[:, 1]

logistic_auc = roc_auc_score(
    validation["target"],
    logistic_probability,
)

print("Best Logistic ROC-AUC:", logistic_auc)

c:\Users\Аделя\Downloads\Лабораторная работа3-ru.ipynb - Colab_files\Intro_ML-lab3\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Logistic ROC-AUC: 0.9156760204081633


## Итог по Logistic Regression

Базовая Logistic Regression использовала стандартное значение `C=1.0` и давала ROC-AUC около `0.884`.

Так как в данных 512 признаков при относительно небольшом размере train, модель могла переобучаться.

Поэтому была усилена L2-регуляризация и подобран параметр `C` по validation.

Проверялись значения:

`0.001, 0.002, 0.003, 0.005, 0.01, 0.03, 0.1, 1.0`

Лучший результат получился при:

- `C=0.003`
- `penalty="l2"`
- `solver="liblinear"`
- `StandardScaler`
- `max_iter=5000`

ROC-AUC вырос примерно с `0.884` до `0.916`.

=> уменьшение `C` усилило регуляризацию и уменьшило переобучение.

In [79]:
blend_results = []

for logistic_weight in [
   0.0,
   0.1,
   0.2,
   0.3,
   0.4,
   0.5,
]:

   extra_weight = 1.0 - logistic_weight

   probability = (
       extra_weight * extra_probability
       +
       logistic_weight * logistic_probability
   )

   auc = roc_auc_score(
       validation["target"],
       probability
   )

   blend_results.append({
       "extra_weight": extra_weight,
       "logistic_weight": logistic_weight,
       "roc_auc": auc,
   })

blend_df = (
   pd.DataFrame(blend_results)
   .sort_values(
       "roc_auc",
       ascending=False,
   )
   .reset_index(drop=True)
)

blend_df

,extra_weight,logistic_weight,roc_auc
0,0.8,0.2,0.930242
1,0.7,0.3,0.929936
2,0.9,0.1,0.929362
3,0.6,0.4,0.929043
4,0.5,0.5,0.927321
5,1.0,0.0,0.925867
